# B2 DeBERTa: train and score on Colab

Spec: `docs/superpowers/specs/2026-11-07-deberta-detector-design.md`. Plan: Task 8 and Task 11 in
`docs/superpowers/plans/2026-11-07-deberta-detector.md`. All logic lives in
`src/models/deberta_run.py`; this notebook only sets parameters and calls it.

**Before the first run**
- Runtime → Change runtime type → **T4 GPU**.
- Add the shared *read-only* Colab R2 token to **your own** Colab Secrets (key icon in the left
  bar) as `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY`, with notebook access on. Get them from
  the lead by password manager or private DM, never in git, an issue, a doc or a group channel.
- Never print the keys, never put them on a `!` command line, never store them in any DVC config file.
- Clear all outputs before committing this notebook (Edit → Clear all outputs).

**Run names** (the score file is named after them)
- Reported runs: `row{1..4}-s{seed}`, seeds 7267, 7268, 7269, with `LIMIT = None` and `VAL_ONLY = False`.
- Weight sweep: `sweep-row2-w{weight}-s7267`, with `VAL_ONLY = True` (never scores the holdouts).
- Smoke test: `ROW = "smoke"` with a `LIMIT`.

**After a disconnect**: reconnect and Run all again with the same parameters. Training resumes
after the last epoch saved on Drive.

In [ ]:
# Parameters: edit these, then Runtime → Run all.
BRANCH = "main"
ROW = "smoke"                          # row1 | row2 | row3 | row4 | smoke
SEED = 7267                            # reported runs: 7267, 7268, 7269
PAIRWISE_WEIGHT = 0.0                  # row1: 0.0; row2/row3: the swept weight
TRAIN_SOURCES = ["bipia", "poisonedrag"]   # row3: ["bipia"]
WITH_SYNTHETIC = False                 # row4, after Task 9 merges
LIMIT = 200                            # None for every reported or sweep run
VAL_ONLY = True                        # True for sweeps and smoke tests: never scores the holdouts
FP16 = True                            # set False if training stops with a non-finite loss
EPOCHS = 3

if LIMIT is not None and ROW != "smoke":
    raise ValueError("LIMIT is for smoke tests only: set ROW = 'smoke' or LIMIT = None")
if ROW == "smoke" and LIMIT is None:
    raise ValueError("ROW = 'smoke' needs a LIMIT")
if WITH_SYNTHETIC:
    raise ValueError("WITH_SYNTHETIC needs Task 9 (phase3/synthetic-tools) merged first")

if ROW == "smoke":
    RUN_NAME = f"smoke-s{SEED}"
elif VAL_ONLY:
    RUN_NAME = f"sweep-{ROW}-w{PAIRWISE_WEIGHT}-s{SEED}"
else:
    RUN_NAME = f"{ROW}-s{SEED}"
print("run name:", RUN_NAME)

In [ ]:
# Drive keeps checkpoints, the model and the score file across disconnects.
import os
from google.colab import drive

drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/cs7267-deberta/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
print("run dir:", RUN_DIR)

In [ ]:
# Code and dependencies. uv installs the locked versions, including torch.
REPO_URL = "https://github.com/maxedmonds07/CS7267-LLM-Injection-Defense.git"
%cd /content
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo fetch --depth 1 origin {BRANCH}
    !git -C /content/repo checkout -q -B {BRANCH} FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q uv
!uv sync --frozen --group deberta --group dev
os.environ["PYTHONPATH"] = "src/data:src/eval:src/models"

In [ ]:
# R2 keys from your Colab Secrets into the environment DVC reads. Nothing is printed.
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('R2_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('R2_SECRET_ACCESS_KEY')
print("R2 credentials loaded from Colab Secrets")

In [ ]:
!uv run --group deberta dvc pull data/processed/dataset.parquet
if _exit_code != 0:
    raise RuntimeError("dvc pull failed: check the Colab Secrets names and that the token is still valid")

In [ ]:
!nvidia-smi
!uv run --group deberta python -c "import torch; assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type > T4 GPU'; print(torch.cuda.get_device_name(0))"
if _exit_code != 0:
    raise RuntimeError("no CUDA GPU in this runtime")
if PAIRWISE_WEIGHT > 0:
    # Until Task 6 merges, a non-zero weight would be silently ignored and the run mislabelled.
    !uv run --group deberta python -c "from deberta import pairwise_margin_loss"
    if _exit_code != 0:
        raise RuntimeError(f"PAIRWISE_WEIGHT > 0 needs Task 6 (pairwise loss) on branch {BRANCH}")

In [ ]:
# Train. Checkpoints go to Drive after every epoch; rerunning resumes after the last one.
import shlex, time

train_args = [
    "--row", ROW, "--seed", str(SEED), "--run-name", RUN_NAME,
    "--model-out", f"{RUN_DIR}/model", "--checkpoint-dir", f"{RUN_DIR}/ckpt",
    "--pairwise-weight", str(PAIRWISE_WEIGHT), "--epochs", str(EPOCHS),
    "--train-sources", *TRAIN_SOURCES,
]
if LIMIT is not None:
    train_args += ["--limit", str(LIMIT)]
if not FP16:
    train_args.append("--no-fp16")
TRAIN_ARGS = shlex.join(train_args)

started = time.time()
!uv run --group deberta python src/models/deberta_run.py train {TRAIN_ARGS}
if _exit_code != 0:
    raise RuntimeError("training failed; a non-finite loss means: set FP16 = False and Run all again")
print(f"train cell took {(time.time() - started) / 60:.1f} min (includes any resumed epochs skipped)")

In [ ]:
# Score. Sweeps and smoke tests score val only; reported runs also score test, MCPTox and MSB.
score_args = ["--model", f"{RUN_DIR}/model", "--out", f"{RUN_DIR}/scores.parquet"]
if VAL_ONLY:
    score_args.append("--val-only")
SCORE_ARGS = shlex.join(score_args)

!uv run --group deberta python src/models/deberta_run.py score {SCORE_ARGS}
if _exit_code != 0:
    raise RuntimeError("scoring failed")

In [ ]:
import json
print(json.dumps(json.load(open(f"{RUN_DIR}/model/run.json")), indent=2))
print(json.dumps({k: v for k, v in json.load(open(f"{RUN_DIR}/model/detector.json")).items()
                  if k in ("selected_epoch", "history")}, indent=2))
print("score file:", f"{RUN_DIR}/scores.parquet")

## Next

- Download `scores.parquet` from the run directory on Drive and save it in the repo as
  `report/deberta_scores/<run name>.parquet`.
- Locally: `make evaluate DETECTOR=deberta SCORES=report/deberta_scores/<run name>.parquet`.
- Once every reported run is in: `make b2-report` (plan Task 11). Do not rerun anything after
  reading the verdict.